In [1]:
import requests
from bs4 import BeautifulSoup
import re
import urllib3
import json
import os

# Vibe Coding by Gemini 都是AI電腦寫的
# 本程式旨在抓取 iTHome 鐵人賽「參賽名單」頁面的詳細資訊。
# 適用於 2022-2025 年版面。

# 忽略因 verify=False 產生的 SSL 警告
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- 參數設定：請修改此處的年度和頁碼 ---
# 設置您想要抓取的 iThome 鐵人賽年份
TARGET_YEAR = 2024
# 設置您想要抓取的名單頁碼（此參數在全頁抓取模式下已不再重要，但保留）
TARGET_PAGE = 1
# -------------------------------------

# --- 常數設定 ---
# 動態 URL 模板：用於參賽名單頁面
URL_TEMPLATE = "https://ithelp.ithome.com.tw/{year}ironman/signup/list?page={page}"

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Connection': 'keep-alive'
}

# -----------------------------------------------------
# 函式：抓取下拉選單 (主題組別) 資料
# -----------------------------------------------------
def get_dropdown_data(soup: BeautifulSoup) -> list:
    """
    抓取下拉選單 (主題組別) 內的資料，並排除第一項「選擇組別」。
    """
    theme_options = []
    # 尋找下拉選單元素
    select_tag = soup.find('select', id='type')
    
    if select_tag:
        options = select_tag.find_all('option')
        # 從第二項 (索引 1) 開始遍歷，排除第一項 (通常是 "選擇組別")
        for option in options[1:]: 
            theme_name = option.get_text(strip=True)
            # 連結值即為 onchange 事件中要導向的 URL 參數
            theme_value = option.get('value')
            
            if theme_name and theme_value:
                theme_options.append({
                    "主題名稱": theme_name,
                    "連結值": theme_value
                })
                
    return theme_options

# -----------------------------------------------------
# 函式：抓取總參賽人數
# -----------------------------------------------------
def get_total_contestant_count(soup: BeautifulSoup) -> int:
    """
    抓取頁面中顯示的總參賽者筆數。
    """
    # 優先尋找 <div class="ppl-num"> 來抓取總人數
    count_element = soup.find('div', class_='ppl-num') 
    
    if count_element:
        # 提取元素內的數字文字
        text = count_element.get_text(strip=True)
        # 使用正規表達式尋找文字中的第一個數字
        match = re.search(r'(\d+)', text)
        if match:
            return int(match.group(1))
    
    # 【Fallback 備用】：如果找不到 ppl-num，則使用 /signup/list 頁面的 header 邏輯
    count_container = soup.find('div', class_='contestants-list__header') 
    
    if count_container:
        # 提取整個容器的文字內容
        text = count_container.get_text(strip=True)
        # 使用更靈活的正規表達式尋找數字（適用於 "共有 X 筆參賽紀錄"）
        match = re.search(r'(\d+)', text)
        if match:
            return int(match.group(1))
            
    return 0 # 如果找不到，回傳 0

# -----------------------------------------------------
# 函式：抓取總分頁數
# -----------------------------------------------------
def get_total_pages(url: str, headers: dict) -> int:
    """
    連線到網頁並從分頁區塊中解析出最大的頁碼數字，即為總分頁數。
    """
    print(f"正在連線至網址以獲取總頁數: {url}")
    try:
        # 禁用 SSL 驗證，並使用全域 headers 模擬瀏覽器
        response = requests.get(url, headers=headers, verify=False)
        response.raise_for_status() 
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # 定位分頁的 HTML 區塊 (class='pagination justify-content-center')
        pagination_div = soup.find('div', class_='pagination justify-content-center')
        
        if pagination_div:
            # 獲取整個分頁區塊的文本內容
            pagination_text = pagination_div.get_text()
            
            # 使用正規表達式從文本中提取所有連續的數字
            page_numbers = re.findall(r'\d+', pagination_text)
            
            # 確保數字列表中沒有空的字串
            page_numbers = [n for n in page_numbers if n.isdigit()]

            if page_numbers:
                # 轉換為整數並找出最大值，即為總頁數
                max_page = max(int(n) for n in page_numbers)
                return max_page
            else:
                # 如果找不到頁碼，但頁面有內容，至少為 1 頁
                return 1
        else:
            # 如果找不到分頁區塊，通常只有 1 頁
            return 1

    except requests.exceptions.RequestException as e:
        print(f"連線或請求發生錯誤: {e}")
        return 1
    except Exception as e:
        print(f"發生未知錯誤: {e}")
        return 1

# -----------------------------------------------------
# 核心函式：抓取單頁參賽名單詳細資料及元數據
# -----------------------------------------------------
def scrape_contestant_list(year: int, page: int):
    """
    從 iThome 鐵人賽的參賽名單頁面抓取所有參賽者的詳細資訊、主題組別和總人數 (僅在 page=1 時有用)。
    
    Returns:
        tuple: (contestants_data, theme_options, total_count, success_status)
    """
    # 根據輸入的 year 和 page 參數建立目標 URL
    target_url = URL_TEMPLATE.format(year=year, page=page)
    
    contestants_data = []
    theme_options = []
    total_count = 0
    success = False # 狀態指示器

    try:
        # 1. 發送 HTTP GET 請求
        response = requests.get(target_url, headers=HEADERS, verify=False)
        response.raise_for_status() 
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 僅在第一頁時抓取主題組別和總人數等元數據
        if page == 1:
            theme_options = get_dropdown_data(soup)
            total_count = get_total_contestant_count(soup)
        
        # 2. 找到所有參賽者資訊區塊
        list_cards = soup.find_all('div', class_='list-card')
        
        if not list_cards:
            # 找到頁面但沒有數據，可能頁碼超出了實際範圍，視為抓取成功（因為連線成功），但沒有數據
            # 注意：在迴圈結束後，抓取到的頁面可能會是空的，但 response 仍然成功
            print(f"警告: 第 {page} 頁未找到參賽者資料 (class='list-card')。URL: {target_url}")
            return [], theme_options, total_count, True # 設為 True 避免重複計入 RequestException 失敗

        
        # 3. 遍歷每個卡片，提取所需資訊
        for card in list_cards:
            # 題目資訊區
            title_link_element = card.find('a', class_='contestants-list__title')
            # 參賽者資訊區
            person_link_element = card.find('a', class_='contestants-list__person')
            # 進度標籤區
            status_label = card.find('label', class_='note')
            # 參賽者圖像元素
            img_element = card.find('img', class_='w-100')

            # ---【報名日期處理】：移除固定前綴文字 ---
            date_element = card.find('div', class_='contestants-list__date')
            raw_date_text = date_element.get_text(strip=True) if date_element else "N/A"
            cleaned_date = raw_date_text.replace('報名日期：', '').strip()
            # ----------------------------------------------------

            # 提取並清理數據
            entry = {
                "年度": year,
                "頁碼": page, # 新增頁碼資訊
                "報名日期": cleaned_date, 
                "主題": card.find('div', class_='tag').get_text(strip=True) if card.find('div', class_='tag') else "N/A",
                "參賽者名稱": person_link_element.find('div', class_='contestants-list__name').get_text(strip=True) if person_link_element and person_link_element.find('div', class_='contestants-list__name') else "N/A",
                "圖像URL": img_element.get('src') if img_element else "N/A",
                "題目": title_link_element.get_text(strip=True) if title_link_element else "N/A",
                "題目URL": title_link_element.get('href') if title_link_element else "N/A",
                "題目簡介": card.find('p', class_='contestants-list__desc').get_text(strip=True) if card.find('p', class_='contestants-list__desc') else "N/A",
                "進度": status_label.get_text(strip=True) if status_label else "N/A"
            }
            
            contestants_data.append(entry)
            
        success = True
        
        # 返回當頁提取的數據、元數據和狀態
        return contestants_data, theme_options, total_count, success

    except requests.exceptions.HTTPError as e:
        print(f"\n[錯誤] 網頁請求失敗: HTTP 錯誤 {e.response.status_code}。頁面: {page}, URL: {target_url}")
    except requests.exceptions.RequestException as e:
        print(f"\n[錯誤] 網頁請求失敗: {e}。頁面: {page}, URL: {target_url}")
    except Exception as e:
        print(f"\n[錯誤] 發生其他錯誤: {e}。頁面: {page}, URL: {target_url}")

    # 如果發生錯誤，返回空數據和失敗狀態
    return [], theme_options, total_count, success # 這裡 success 為 False

# -----------------------------------------------------
# 主執行區塊
# -----------------------------------------------------
def main():
    
    # 1. 構建用於獲取總頁數的第一頁 URL
    first_page_url = URL_TEMPLATE.format(year=TARGET_YEAR, page=1)
    
    # 2. 獲取總頁數
    total_pages = get_total_pages(first_page_url, HEADERS)
    
    # 初始化聚合容器和計數器
    all_contestants_data = []
    theme_options = []
    total_count = 0
    success_count = 0 
    failure_count = 0
    failed_urls = []
    
    # 3. 先抓取第 1 頁數據，以獲取總人數 (total_count) 和主題 (theme_options)
    # 移除原有的初始化標頭和訊息
    
    # 執行第 1 頁抓取
    page1_data, theme_options, total_count_from_page1, success_page1 = scrape_contestant_list(TARGET_YEAR, 1)
    
    # 檢查第 1 頁抓取結果
    if success_page1:
        # P1 的 metadata 數據必須保留
        total_count = total_count_from_page1 # 更新總人數
        
        # 移除原有的第 1 頁進度列印
    else:
        # P1 失敗，先將其 URL 記錄下來
        failed_urls.append(first_page_url)
        # 由於 scrape_contestant_list 已經打印過錯誤，這裡只記錄失敗次數

    
    # 4. 打印使用者要求的乾淨標頭 (總人數已知)
    print("\n" + "=="*25)
    print(f"iTHome 鐵人賽 {TARGET_YEAR} 年")
    print(f"報名總人數: {total_count}") 
    print(f"【開始全分頁抓取作業】總分頁數預計為: {total_pages}")
    print("=="*25)

    # 5. 循環抓取所有頁面數據 (從第 1 頁開始，如果需要的話)
    start_page = 1
    
    # --- 修正 P1 進度輸出邏輯，確保動態覆蓋 ---
    if success_page1:
        all_contestants_data.extend(page1_data)
        success_count += 1
        
        # 使用 \r 和 end='' 打印 P1 進度，讓後續的 P2 可以覆蓋它
        output_line = f"\r✅ 成功抓取 {len(page1_data)} 筆數據。 [1/{total_pages}]"
        print(output_line.ljust(80), end='') 
        
        start_page = 2 if total_pages > 1 else 2
    else:
        # P1 失敗的進度輸出 (永久保留)
        print(f"❌ 抓取失敗，已跳過此頁。 [1/{total_pages}]") 
        start_page = 2
    # ------------------------------------------
        
    # 循環抓取所有頁面數據 (從 start_page 開始)
    # 如果 total_pages <= 1 且 start_page 為 2，range(2, 2) 或 range(2, 1) 不會執行，這是正確的
    for page in range(start_page, total_pages + 1): 
        # 執行單頁資料抓取，現在返回 4 個值
        current_page_data, _, _, success = scrape_contestant_list(TARGET_YEAR, page)
        
        target_url = URL_TEMPLATE.format(year=TARGET_YEAR, page=page)
        
        # 處理結果
        if success:
            all_contestants_data.extend(current_page_data)
            success_count += 1
            # 簡潔的進度輸出，使用 \r 覆蓋前一行
            output_line = f"\r✅ 成功抓取 {len(current_page_data)} 筆數據。 [{page}/{total_pages}]"
            print(output_line.ljust(80), end='') 
        else:
            failure_count += 1
            failed_urls.append(target_url)
            # 簡潔的進度輸出 (失敗時使用 \n 換行，這樣錯誤訊息會永久保留在終端機中)
            print(f"\n❌ 抓取失敗，已跳過此頁。 [{page}/{total_pages}]") 

    # 循環結束後，輸出一個換行符號，確保後續的摘要報告從新的一行開始
    print()

    if not all_contestants_data and total_count == 0:
        print("\n程式結束，因為沒有抓取到任何有效的數據。")
        return

    # 6. 打印最終摘要 (符合使用者要求的格式)
    print("\n" + "=="*25)
    print(f"【抓取摘要】")
    print(f"完成進度: {success_count + failure_count}/{total_pages}")
    print(f"成功頁面: {success_count}")
    print(f"失敗頁面: {failure_count}")
    if failed_urls:
        print("失敗的頁面 URL 列表:")
        for url in failed_urls:
            print(f" - {url}")
    else:
        print("失敗的頁面 URL 列表: 無")
    print("=="*25)

    # 7. 組合最終 JSON 結構 (使用更新後的 total_count)
    output_key = f"it 鐵人賽 {TARGET_YEAR} 年 - 參賽名單"
    final_output = {
        output_key: {
            "總參賽人數_所有組別": total_count,
            "總分頁數": total_pages,
            "實際抓取筆數": len(all_contestants_data),
            "主題組別_下拉選單": theme_options,
            "參賽者名單_完整列表": all_contestants_data
        }
    }
    
    # 8. 檔案命名與儲存
    output_filename = f"ironman_{TARGET_YEAR}_list_all_pages.json" # 更改檔案名稱以區分
    
    try:
        # 生成 JSON 字串
        json_output = json.dumps(final_output, indent=4, ensure_ascii=False)
        
        # 寫入檔案
        with open(output_filename, 'w', encoding='utf-8') as f:
            f.write(json_output)
        
        # 輸出結果和儲存路徑
        print("\n" + "=="*25)
        print("【JSON 數據抓取與儲存完成】")
        print(f"最終儲存筆數: {len(all_contestants_data)}")
        
        file_path = os.path.abspath(output_filename)
        print(f"\n✅ JSON 檔案已成功儲存至：\n{file_path}")
        print("=="*25)

    except IOError as e:
        if "Permission denied" in str(e):
             print("\n[檔案錯誤] 寫入失敗: 權限不足 (Permission denied)。請檢查檔案寫入權限。")
        else:
             print(f"\n[錯誤] 檔案寫入失敗: {e}")
    except Exception as e:
        print(f"\n[錯誤] 發生其他錯誤: {e}")

if __name__ == "__main__":
    main()


正在連線至網址以獲取總頁數: https://ithelp.ithome.com.tw/2024ironman/signup/list?page=1

iTHome 鐵人賽 2024 年
報名總人數: 1064
【開始全分頁抓取作業】總分頁數預計為: 107
✅ 成功抓取 4 筆數據。 [107/107]                                                        

【抓取摘要】
完成進度: 107/107
成功頁面: 107
失敗頁面: 0
失敗的頁面 URL 列表: 無

【JSON 數據抓取與儲存完成】
最終儲存筆數: 1064

✅ JSON 檔案已成功儲存至：
c:\Users\Hsu\Desktop\mylab_2025\vibe-coding\exchange-rate-webapp\ironman_2024_list_all_pages.json
